# LIP-TRAC: WER and CER on the held-out clips

Computes the word error rate and character error rate reported in the paper, for the saved
weights `models/FINAL_LIPMODEL.weights.h5` (full precision) and optionally for the quantized
TensorFlow Lite model that runs on the Raspberry Pi.

* WER = word-level Levenshtein distance summed over all clips, divided by the total number of
  reference words (corpus level, as `jiwer.wer` computes it). CER is the same over characters.
* The original evaluation decoded with Keras' CTC beam search (beam width 100) and passed the
  full padded length of 290 frames for every clip. The Pi scripts decode greedily. Both are
  reported here.
* The shuffle in the original run was unseeded, so the held-out set is re-drawn with `SEED`;
  the clips are not the same 455 as in the reported run. To evaluate the *same* split used for
  training in `final_run.ipynb`, keep `SEED` identical there and here.

Nothing in this notebook has been executed in the repository; outputs are intentionally absent.

In [ ]:
import os
import csv
import random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
from jiwer import wer, cer
from matplotlib import pyplot as plt

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Paths. DATA_DIR holds pairs <clip>.mp4 / <clip>.txt copied from LRS2 "main"
# (see notebooks/data/trainFileGen.ipynb). LRS2 itself is not in this repository.
DATA_DIR = Path(os.environ.get("LIPTRAC_DATA", "../../data/trainFiles1"))
CASCADE_PATH = Path("../../assets/haarcascade_frontalface_default.xml")
WEIGHTS_PATH = Path("../../models/FINAL_LIPMODEL.weights.h5")
OUT_DIR = Path("runs/final_run")

# Data and split. The final run used 1,139 clips, batches of 4, and a 60/40 split
# taken over batches (171 training batches = 684 clips; 114 held-out batches = 455 clips,
# the last batch holding 3). The original shuffle was unseeded, so the exact clips in each
# half of the reported run cannot be reconstructed; SEED fixes a split for this notebook.
SEED = 1139
BATCH_SIZE = 4
TRAIN_FRACTION = 0.6

# Input geometry (fixed by the model: 290 frames of 40 x 120 grayscale mouth crops).
IMAGE_SIZE = (40, 120)   # (height, width)
MAX_FRAMES = 290         # 2 * MAX_CHARS, as in the original notebooks
MAX_CHARS = 145

In [ ]:
TFLITE_PATH = Path("../../models/finalModel.tflite")   # set to None to skip the quantized model

## Data pipeline (identical to `final_run.ipynb`)

In [ ]:
# Characters -> integers and back. Index 0 is the padding/OOV token; the Dense layer adds
# one more class for the CTC blank, giving 29 classes.
VOCAB = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ ")
char_to_num = tf.keras.layers.StringLookup(vocabulary=VOCAB, oov_token="")
num_to_char = tf.keras.layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)
NUM_CLASSES = char_to_num.vocabulary_size() + 1

face_cascade = cv2.CascadeClassifier(str(CASCADE_PATH))
assert not face_cascade.empty(), f"could not load {CASCADE_PATH}"
last_known_crop = (0, 0, 160, 150)


def detect_faces(gray):
    return face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30))


def crop_mouth(frame):
    """Fixed crop relative to the largest detected face; reuses the last crop when no face is found.

    Rows: from 65% of the face height to the bottom of the box.
    Columns: from x + 0.05 * w to 0.95 * (x + w). This is exactly what the training notebooks
    and pi/cropFuncs.py do; note that the right edge is 95% of the right coordinate, not
    x + 0.95 * w."""
    global last_known_crop
    rects = detect_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    largest = (0, 0, 0, 0)
    for (x, y, w, h) in rects:
        if w * h > largest[2] * largest[3]:
            largest = (x, y, w, h)
    if largest == (0, 0, 0, 0):
        largest = last_known_crop
    last_known_crop = largest
    x, y, w, h = largest
    return frame[y + int(0.65 * h): y + h, x + int(0.05 * w): int(0.95 * (x + w))]


def load_video(path):
    """Mouth crops for every frame, resized to IMAGE_SIZE, grayscale, standardized per clip."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    first_shape = None
    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):
        ok, frame = cap.read()
        if not ok or frame is None or frame.shape[0] == 0:
            continue
        if first_shape is None:
            first_shape = frame.shape
        if frame.shape != first_shape:
            continue
        mouth = crop_mouth(frame)
        mouth = cv2.resize(mouth, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
        frames.append(tf.image.rgb_to_grayscale(mouth))
    cap.release()
    # Same arithmetic as the original notebooks: the mean is taken over the uint8 frames
    # (so it is truncated to an integer) and the standard deviation in float32.
    mean = tf.math.reduce_mean(frames, keepdims=True)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32), keepdims=True)
    frames = tf.cast(frames, tf.float32)
    return (frames - tf.cast(mean, tf.float32)) / std


def number_to_words(num):
    if num == 0:
        return "ZERO"
    ones = ["", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
    tens = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
    teens = ["ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
    words = ""
    if num >= 1000:
        words += ones[num // 1000] + " thousand "
        num %= 1000
    if num >= 100:
        words += ones[num // 100] + " hundred "
        num %= 100
    if 10 <= num <= 19:
        words += teens[num - 10] + " "
        num = 0
    elif num >= 20:
        words += tens[num // 10] + " "
        num %= 10
    if 1 <= num <= 9:
        words += ones[num] + " "
    return words.strip().upper()


def load_transcript(path):
    """LRS2 transcript files start with a line 'Text:  THE WORDS ...'. Numerals become words."""
    with open(path, "r") as f:
        words = f.readline().split()[1:]
    words = [number_to_words(int(w)) if w.isnumeric() else w for w in words]
    text = " ".join(" ".join(words).split()).upper()
    return char_to_num(tf.strings.unicode_split(text, input_encoding="UTF-8"))


def load_clip(path_tensor):
    stem = Path(bytes.decode(path_tensor.numpy())).stem
    return load_video(DATA_DIR / f"{stem}.mp4"), load_transcript(DATA_DIR / f"{stem}.txt")


def process_clip(path):
    return tf.py_function(load_clip, [path], (tf.float32, tf.int64))


def decode_labels(label_ids):
    return tf.strings.reduce_join(num_to_char(label_ids)).numpy().decode("utf-8").strip()

## Split

In [ ]:
clip_paths = sorted(str(p) for p in DATA_DIR.glob("*.mp4"))
assert clip_paths, f"no clips found in {DATA_DIR.resolve()}"
print("clips:", len(clip_paths))

dataset = tf.data.Dataset.from_tensor_slices(clip_paths)
dataset = dataset.shuffle(len(clip_paths), seed=SEED, reshuffle_each_iteration=False)
dataset = dataset.map(process_clip)
dataset = dataset.padded_batch(BATCH_SIZE, padded_shapes=([MAX_FRAMES, None, None, None], [MAX_CHARS]))
dataset = dataset.prefetch(tf.data.AUTOTUNE)

num_batches = len(dataset)
num_train_batches = int(num_batches * TRAIN_FRACTION)
train_ds = dataset.take(num_train_batches)
test_ds = dataset.skip(num_train_batches)
print("batches:", num_batches, " train batches:", num_train_batches, " held-out batches:", num_batches - num_train_batches)

# Record the split parameters next to the run.
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "split_seed.txt", "w") as f:
    f.write(f"seed={SEED} batch_size={BATCH_SIZE} train_fraction={TRAIN_FRACTION} clips={len(clip_paths)}\n")

## Model

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, TimeDistributed, Flatten, Bidirectional, GRU, Dropout, Dense

INPUT_SHAPE = (MAX_FRAMES, IMAGE_SIZE[0], IMAGE_SIZE[1], 1)


def create_model(gru_units=256):
    return Sequential([
        Conv3D(32, 3, input_shape=INPUT_SHAPE, padding="same", activation="relu", name="conv1"),
        MaxPooling3D((1, 2, 2), name="maxPool1"),
        Conv3D(64, 3, padding="same", activation="relu", name="conv2"),
        MaxPooling3D((1, 2, 2), name="maxPool2"),
        Conv3D(96, 3, padding="same", activation="relu", name="conv3"),
        MaxPooling3D((1, 2, 2), name="maxPool3"),
        TimeDistributed(Flatten()),
        Bidirectional(GRU(gru_units, kernel_initializer="orthogonal", return_sequences=True)),
        Dropout(0.5),
        Bidirectional(GRU(gru_units, kernel_initializer="orthogonal", return_sequences=True)),
        Dropout(0.5),
        Dense(NUM_CLASSES, kernel_initializer="he_normal", activation="softmax"),
    ])


def ctc_loss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], "int64")
    input_len = tf.cast(tf.shape(y_pred)[1], "int64") * tf.ones((batch_len, 1), "int64")
    label_len = tf.cast(tf.shape(y_true)[1], "int64") * tf.ones((batch_len, 1), "int64")
    return tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_len, label_len)


model = create_model(256)
model.summary()   # expected total: 12,875,325 parameters

In [ ]:
model.load_weights(str(WEIGHTS_PATH))
print("loaded", WEIGHTS_PATH)

## Decoding and metrics

In [ ]:
def decode_batch(y_pred, greedy=False, beam_width=100):
    """CTC decoding of a batch of per-frame distributions. The original evaluation code passed the
    full padded length (MAX_FRAMES) for every clip and used Keras' beam search (beam width 100);
    the Raspberry Pi scripts decode greedily."""
    lengths = [MAX_FRAMES] * int(y_pred.shape[0])
    decoded = tf.keras.backend.ctc_decode(y_pred, input_length=lengths, greedy=greedy, beam_width=beam_width, top_paths=1)[0][0]
    return [decode_labels(row) for row in decoded.numpy()]


def evaluate(model_or_predict, ds, greedy=False):
    """Corpus-level WER and CER over every clip in ds, plus the per-clip pairs."""
    refs, hyps = [], []
    for frames, labels in ds:
        y_pred = model_or_predict(frames)
        hyps.extend(decode_batch(y_pred, greedy=greedy))
        refs.extend(decode_labels(row) for row in labels.numpy())
    return {"wer": wer(refs, hyps), "cer": cer(refs, hyps), "clips": len(refs)}, list(zip(refs, hyps))

## Full-precision model

In [ ]:
results = {}
results["keras_beam"], pairs = evaluate(model.predict, test_ds, greedy=False)
results["keras_greedy"], _ = evaluate(model.predict, test_ds, greedy=True)
for k, v in results.items():
    print(f"{k:14s} WER {100 * v['wer']:.1f}%  CER {100 * v['cer']:.1f}%  clips {v['clips']}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "heldout_predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["reference", "prediction"])
    w.writerows(pairs)

## Quantized TensorFlow Lite model

In [ ]:
# Optional: the same clips through the quantized TensorFlow Lite model (one clip per invocation,
# as on the Raspberry Pi). This is the on-device accuracy the paper does not report.
if TFLITE_PATH is not None and Path(TFLITE_PATH).exists():
    interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
    interpreter.allocate_tensors()
    in_index = interpreter.get_input_details()[0]["index"]
    out_index = interpreter.get_output_details()[0]["index"]

    def tflite_predict(frames):
        outs = []
        for clip in frames.numpy().astype(np.float32):
            interpreter.set_tensor(in_index, clip[None, ...])
            interpreter.invoke()
            outs.append(interpreter.get_tensor(out_index)[0])
        return tf.constant(np.stack(outs))

    results["tflite_greedy"], _ = evaluate(tflite_predict, test_ds, greedy=True)
    results["tflite_beam"], _ = evaluate(tflite_predict, test_ds, greedy=False)
    for k in ("tflite_greedy", "tflite_beam"):
        v = results[k]
        print(f"{k:14s} WER {100 * v['wer']:.1f}%  CER {100 * v['cer']:.1f}%  clips {v['clips']}")

## Doubled-letter errors

In [ ]:
# The paper's error analysis: how often a doubled letter in the reference comes out as a single letter.
import re
doubled = sum(1 for ref, _ in pairs for w in ref.split() if re.search(r"(.)\1", w))
merged = sum(1 for ref, hyp in pairs for w in ref.split()
             if re.search(r"(.)\1", w) and re.sub(r"(.)\1", r"\1", w) in hyp.split() and w not in hyp.split())
print(f"reference words with a doubled letter: {doubled}; predicted with the pair merged: {merged}")